## CNN Feature Extraction + Random Forest for ICBHI Respiratory Sounds

This notebook combines a **pretrained ResNet18 feature extractor** with a **Random Forest classifier** for respiratory-cycle classification using the ICBHI dataset.

The workflow is to load the dataset metadata, segment respiratory cycles, create log-Mel spectrograms, convert each spectrogram into an image, extract a fixed-length feature vector with ResNet18, train a Random Forest on the four respiratory categories, and evaluate the predictions as **Normal vs. Abnormal**.

**Hint:** Run the notebook from top to bottom. Check the dataset paths and ensure the required audio files and annotation files are available before processing the full dataset.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import cv2
from tqdm import tqdm

import torch
import torchvision.models as models

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

### Set the Audio Directory

`DATA_DIR` points to the folder containing the ICBHI WAV recordings. The metadata file is loaded later from `../datasets/split_metadata.csv`.

**Hint:** These paths are relative to the notebook's current working directory. If a file cannot be found, check `Path.cwd()` and adjust the paths rather than changing the dataset split.

In [2]:
DATA_DIR = "../datasets/audios"


### Map Crackle/Wheeze Annotations to Four Classes

`get_label()` combines the crackle and wheeze indicators into four labels:

- `0`: Normal (neither crackle nor wheeze)
- `1`: Crackle only
- `2`: Wheeze only
- `3`: Both crackle and wheeze

The notebook later groups label `0` as normal and labels `1–3` as abnormal for binary evaluation.

**Hint:** Verify that annotation columns contain the expected binary values (`0` or `1`).

In [3]:
# Label Mapping

def get_label(crackle, wheeze):
    if crackle == 0 and wheeze == 0:
        return 0  # normal
    elif crackle == 1 and wheeze == 0:
        return 1  # crackle
    elif crackle == 0 and wheeze == 1:
        return 2  # wheeze
    else:
        return 3  # both


In [4]:
# Load and Segment Audio

def load_segments(wav_path, txt_path, sr=16000):
    audio, sr = librosa.load(wav_path, sr=sr)

    ann = pd.read_csv(txt_path, sep='\t', header=None)
    ann.columns = ['start', 'end', 'crackle', 'wheeze']

    segments = []

    for _, row in ann.iterrows():
        start = int(row['start'] * sr)
        end = int(row['end'] * sr)

        segment = audio[start:end]
        label = get_label(row['crackle'], row['wheeze'])

        segments.append((segment, label))

    return segments


### Convert Each Cycle to a Log-Mel Spectrogram

`to_mel()` computes a power Mel spectrogram with 32 Mel bands, an FFT size of 1024, a hop length of 256, and a frequency range of 50–2,000 Hz. It then converts the power values to decibels and standardizes the resulting matrix.

**Hint:** Each cycle can have a different duration, so its spectrogram can have a different number of time frames. The next step resizes it to a fixed image size.

In [5]:
# Mel Spectrogram

def to_mel(segment, sr=16000):
    mel = librosa.feature.melspectrogram(
        y=segment,
        sr=sr,
        n_mels=32,
        n_fft=1024,
        hop_length=256,
        win_length=1024,
        fmin=50,
        fmax=2000,
        power=2.0
    )

    mel_db = librosa.power_to_db(mel)

    # normalize
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

    return mel_db


### Resize the Spectrogram for ResNet18

`prepare_image()` resizes the spectrogram to `224 × 224` and repeats it across three channels so it has the image shape expected by a standard pretrained ResNet18.

**Hint:** The three channels are copies of the same spectrogram, not three independent audio features.

In [6]:
def prepare_image(spec):
    spec = cv2.resize(spec, (224, 224))
    spec = np.stack([spec, spec, spec], axis=-1)
    return spec


### Load Pretrained ResNet18 as a Feature Extractor

The final classification layer is removed from ResNet18, leaving the convolutional network and global pooling layer. The model is placed in evaluation mode, and its weights are not trained in this notebook.

**Hint:** The `pretrained=True` argument may produce a deprecation warning in newer torchvision versions; if necessary, use the current `ResNet18_Weights` API while preserving the intended pretrained weights.

In [7]:
# Load CNN (ResNet18)

model = models.resnet18(pretrained=True)
model = torch.nn.Sequential(*list(model.children())[:-1])
model.eval()


D:\.JKU CS courses\Master Thesis\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
D:\.JKU CS courses\Master Thesis\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Con

### Extract Fixed-Length CNN Features

`extract_features()` converts an image from HWC layout to CHW, adds a batch dimension, and passes it through ResNet18 without gradient calculation. The output is flattened into a feature vector for the Random Forest.


In [8]:
# Feature Extraction

def extract_features(image):
    image = torch.tensor(image).permute(2, 0, 1).unsqueeze(0).float()

    with torch.no_grad():
        feat = model(image)

    return feat.flatten().numpy()


### Build Training and Test Feature Sets

For each row in `split_metadata.csv`, this cell locates the source recording, loads it at 16 kHz, extracts the cycle using its start/end timestamps, skips missing files and very short segments, then computes the spectrogram image and ResNet18 feature vector.

Rows are assigned to `X_train`/`y_train` or `X_test`/`y_test` according to the provided official data `split` column.


In [9]:
# Build Dataset

df = pd.read_csv("../datasets/split_metadata.csv")

X_train, y_train = [], []
X_test, y_test = [], []

for _, row in tqdm(df.iterrows(), total=len(df)):

    split = row["split"]

    file = row["source_file"]
    file = file if file.endswith(".wav") else file + ".wav"
    wav_path = os.path.join(DATA_DIR, file)
    
    if not os.path.exists(wav_path):
        continue

    # load full audio
    audio, sr = librosa.load(wav_path, sr=16000)

    # extract segment
    start = int(row["start"] * sr)
    end = int(row["end"] * sr)

    segment = audio[start:end]

    if len(segment) < 1000:
        continue

    label = get_label(row["crackle"], row["wheeze"])

    # feature pipeline
    mel = to_mel(segment)
    img = prepare_image(mel)
    feat = extract_features(img)

    # assign based on official split
    if split == "train":
        X_train.append(feat)
        y_train.append(label)
    else:
        X_test.append(feat)
        y_test.append(label)

100%|██████████████████████████████████████████████████████████████████████████████| 6898/6898 [24:58<00:00,  4.60it/s]


### Convert Lists to NumPy Arrays

The feature and label lists are converted into NumPy arrays so they can be passed to scikit-learn.

**Hint:** Check that `X_train` and `X_test` are two-dimensional arrays and that their row counts match `y_train` and `y_test`, respectively.

In [10]:
X_train = np.array(X_train)
y_train = np.array(y_train)

X_test = np.array(X_test)
y_test = np.array(y_test)

### Train the Random Forest Classifier

The Random Forest uses 300 trees, a maximum tree depth of 20, and `random_state=42` for reproducibility. It learns to predict the four respiratory categories from the pretrained CNN feature vectors.

**Hint:** ResNet18 is used only to extract features here; the Random Forest is the trainable classifier. Avoid tuning settings based on repeated evaluations of the official test set.

In [11]:
# Train Random Forest

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    random_state=42
)

rf.fit(X_train, y_train)



,n_estimators,300
,criterion,'gini'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [12]:
y_pred = rf.predict(X_test)

In [13]:
# Evaluation

from sklearn.metrics import confusion_matrix

y_test_binary = (y_test != 0).astype(int)
y_pred_binary = (y_pred != 0).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test_binary,
    y_pred_binary
).ravel()

In [14]:
SE = tp / (tp + fn)
SP = tn / (tn + fp)

HS = (2 * SE * SP) / (SE + SP)

print(f"Sensitivity (SE): {SE:.4f}")
print(f"Specificity (SP): {SP:.4f}")
print(f"Harmonic Score (HS): {HS:.4f}")

Sensitivity (SE): 0.2022
Specificity (SP): 0.7821
Harmonic Score (HS): 0.3213


In [15]:
# 13. Save Model

MODEL_DIR = Path("../models/resnet18_features+random_forest/")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = MODEL_DIR / "resnet18_features_random_forest.pt"

checkpoint = {
    "model": rf,
    "model_type": "RandomForestClassifier",
    "n_estimators": 300,
    "max_depth": 20,
    "random_state": 42,
    "feature_extractor": "pretrained ResNet18",
    "feature_type": "normalized log-Mel spectrogram image features",
    "n_mels": 32,
    "sample_rate": 16000,
    "class_names": {
        0: "Normal",
        1: "Crackle only",
        2: "Wheeze only",
        3: "Crackle and wheeze",
    },
}

torch.save(checkpoint, CHECKPOINT_PATH)
print(f"Model checkpoint saved to: {CHECKPOINT_PATH.resolve()}")

Model checkpoint saved to: D:\.JKU CS courses\Master Thesis\models\resnet18_features+random_forest\resnet18_features_random_forest.pt
